<a href="https://colab.research.google.com/github/Hadeel-93/HW-8/blob/main/HW_8_Q2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Download CIFAR-10, convert to tensor and normalize

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2470, 0.2435, 0.2616]
    ),
])

data_path = "./data-cifar10/"

trainset = datasets.CIFAR10(root=data_path, train=True, download=True, transform=transform)
valset   = datasets.CIFAR10(root=data_path, train=False, download=True, transform=transform)

trainloader = DataLoader(trainset, batch_size=128, shuffle=True)
valloader   = DataLoader(valset,   batch_size=128, shuffle=False)

Using device: cuda


100%|██████████| 170M/170M [12:49<00:00, 221kB/s]


In [9]:
import random
import numpy as np
import torch

# Check GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not available. In Colab, select Runtime > Change runtime type > GPU."
    )

device = torch.device("cuda")

print("Using device:", device)
print("GPU:", torch.cuda.get_device_name(0))

# Reproducibility
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# GPU performance settings
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

torch.cuda.empty_cache()

Using device: cuda
GPU: Tesla T4


In [10]:
# CIFAR-10 dataset path
data_path = "/content/data-cifar10/"

# Image transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2470, 0.2435, 0.2616]
    )
])

# Training dataset
train_dataset = datasets.CIFAR10(
    root=data_path,
    train=True,
    download=True,
    transform=transform
)

# Evaluation dataset
eval_dataset = datasets.CIFAR10(
    root=data_path,
    train=False,
    download=True,
    transform=transform
)

print("Training images:", len(train_dataset))
print("Evaluation images:", len(eval_dataset))

classes = train_dataset.classes
print("Classes:", classes)


Training images: 50000
Evaluation images: 10000
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [11]:
BATCH_SIZE = 128
NUM_WORKERS = 2

def create_loaders(seed=42):

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        generator=generator
    )

    eval_loader = DataLoader(
        eval_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True
    )

    return train_loader, eval_loader


train_loader, eval_loader = create_loaders()

print("Training batches:", len(train_loader))
print("Evaluation batches:", len(eval_loader))


Training batches: 391
Evaluation batches: 79


In [12]:
class ResidualBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1,
        use_batchnorm=False,
        dropout_p=0.0
    ):
        super().__init__()

        self.use_batchnorm = use_batchnorm
        self.dropout_p = dropout_p

        # First convolution
        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=not use_batchnorm
        )

        self.bn1 = (
            nn.BatchNorm2d(out_channels)
            if use_batchnorm else nn.Identity()
        )

        # Second convolution
        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=not use_batchnorm
        )

        self.bn2 = (
            nn.BatchNorm2d(out_channels)
            if use_batchnorm else nn.Identity()
        )

        # Dropout is applied between the two convolutions
        self.dropout = (
            nn.Dropout2d(p=dropout_p)
            if dropout_p > 0 else nn.Identity()
        )

        # Shortcut connection
        if stride != 1 or in_channels != out_channels:

            shortcut_layers = [
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=not use_batchnorm
                )
            ]

            if use_batchnorm:
                shortcut_layers.append(
                    nn.BatchNorm2d(out_channels)
                )

            self.shortcut = nn.Sequential(*shortcut_layers)

        else:
            self.shortcut = nn.Identity()

    def forward(self, x):

        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = F.relu(out)

        out = self.dropout(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity

        out = F.relu(out)

        return out


In [13]:
class ResNet10(nn.Module):

    def __init__(self, use_batchnorm=False, dropout_p=0.0):
        super().__init__()

        self.use_batchnorm = use_batchnorm
        self.dropout_p = dropout_p

        # Initial convolution
        self.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=not use_batchnorm
        )

        self.bn1 = (
            nn.BatchNorm2d(64)
            if use_batchnorm else nn.Identity()
        )

        # Five stages, two residual blocks per stage
        self.in_channels = 64

        self.layer1 = self.make_layer(
            64, num_blocks=2, stride=1
        )

        self.layer2 = self.make_layer(
            128, num_blocks=2, stride=2
        )

        self.layer3 = self.make_layer(
            256, num_blocks=2, stride=2
        )

        self.layer4 = self.make_layer(
            512, num_blocks=2, stride=2
        )

        self.layer5 = self.make_layer(
            512, num_blocks=2, stride=1
        )

        # Global average pooling
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Output layer for 10 CIFAR-10 classes
        self.fc = nn.Linear(512, 10)

    def make_layer(self, out_channels, num_blocks, stride):

        layers = []

        layers.append(
            ResidualBlock(
                self.in_channels,
                out_channels,
                stride=stride,
                use_batchnorm=self.use_batchnorm,
                dropout_p=self.dropout_p
            )
        )

        self.in_channels = out_channels

        for _ in range(1, num_blocks):

            layers.append(
                ResidualBlock(
                    self.in_channels,
                    out_channels,
                    stride=1,
                    use_batchnorm=self.use_batchnorm,
                    dropout_p=self.dropout_p
                )
            )

        return nn.Sequential(*layers)

    def forward(self, x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)

        x = self.avg_pool(x)

        x = torch.flatten(x, 1)

        x = self.fc(x)

        return x


In [14]:
test_model = ResNet10(
    use_batchnorm=False,
    dropout_p=0.0
).to(device)

# Count residual blocks
residual_block_count = sum(
    1 for module in test_model.modules()
    if isinstance(module, ResidualBlock)
)

# Count trainable parameters
def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

# Test forward pass
sample_images, sample_labels = next(iter(train_loader))
sample_images = sample_images.to(device)

with torch.no_grad():
    sample_outputs = test_model(sample_images)

print("Residual blocks:", residual_block_count)
print("Input shape:", sample_images.shape)
print("Output shape:", sample_outputs.shape)
print("Trainable parameters:", count_parameters(test_model))

assert residual_block_count == 10
assert sample_outputs.shape[1] == 10

print("Model check successful!")


Residual blocks: 10
Input shape: torch.Size([128, 3, 32, 32])
Output shape: torch.Size([128, 10])
Trainable parameters: 20608394
Model check successful!


In [15]:
def train_one_epoch(model, loader, optimizer, criterion, scaler):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

        predicted = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy


def evaluate_model(model, loader, criterion):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                outputs = model(images)
                loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predicted = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    eval_loss = running_loss / total
    eval_accuracy = correct / total

    return eval_loss, eval_accuracy


In [17]:
import os
import pandas as pd

NUM_EPOCHS = 300
LEARNING_RATE = 0.001

os.makedirs("/content/resnet_results", exist_ok=True)


def run_experiment(
    experiment_name,
    use_batchnorm=False,
    dropout_p=0.0,
    weight_decay=0.0
):

    print("\n" + "=" * 70)
    print("Starting:", experiment_name)
    print("=" * 70)

    # Reset seed for comparable initialization
    set_seed(SEED)

    # Fresh DataLoaders for each experiment
    train_loader, eval_loader = create_loaders(SEED)

    # Create a fresh ResNet-10
    model = ResNet10(
        use_batchnorm=use_batchnorm,
        dropout_p=dropout_p
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=weight_decay
    )

    scaler = torch.amp.GradScaler("cuda")

    # Store metrics
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "eval_loss": [],
        "eval_accuracy": [],
        "epoch_time": []
    }

    # Start total training time
    torch.cuda.synchronize()
    total_start = time.time()

    for epoch in range(NUM_EPOCHS):

        epoch_start = time.time()

        # Training
        train_loss, train_accuracy = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            scaler
        )

        # Evaluation
        eval_loss, eval_accuracy = evaluate_model(
            model,
            eval_loader,
            criterion
        )

        torch.cuda.synchronize()

        epoch_time = time.time() - epoch_start

        # Save metrics
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["eval_loss"].append(eval_loss)
        history["eval_accuracy"].append(eval_accuracy)
        history["epoch_time"].append(epoch_time)

        # Print results every epoch
        print(
            f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
            f"| Time: {epoch_time:.2f}s "
            f"| Train Loss: {train_loss:.4f} "
            f"| Train Acc: {train_accuracy * 100:.2f}% "
            f"| Eval Loss: {eval_loss:.4f} "
            f"| Eval Acc: {eval_accuracy * 100:.2f}%"
        )

    torch.cuda.synchronize()
    total_training_time = time.time() - total_start

    # Count parameters
    parameters = count_parameters(model)

    # Save checkpoint
    safe_name = experiment_name.lower().replace(" ", "_")

    model_path = (
        f"/content/resnet_results/{safe_name}.pth"
    )

    torch.save(model.state_dict(), model_path)

    # Save training history
    history_df = pd.DataFrame(history)

    history_path = (
        f"/content/resnet_results/{safe_name}_history.csv"
    )

    history_df.to_csv(history_path, index=False)

    # Calculate checkpoint size
    model_size_mb = os.path.getsize(model_path) / (1024 ** 2)

    # Final results
    results = {
        "Experiment": experiment_name,
        "Training Time (min)": total_training_time / 60,
        "Final Train Loss": history["train_loss"][-1],
        "Final Train Accuracy (%)": history["train_accuracy"][-1] * 100,
        "Final Eval Loss": history["eval_loss"][-1],
        "Final Eval Accuracy (%)": history["eval_accuracy"][-1] * 100,
        "Parameters": parameters,
        "Model Size (MB)": model_size_mb,
        "Weight Decay": weight_decay,
        "Dropout": dropout_p,
        "Batch Normalization": use_batchnorm
    }

    print("\n" + "=" * 70)
    print("FINAL RESULTS:", experiment_name)
    print("=" * 70)

    for key, value in results.items():
        print(f"{key}: {value}")

    # Plot loss
    plt.figure(figsize=(9, 5))
    plt.plot(history["train_loss"], label="Training Loss")
    plt.plot(history["eval_loss"], label="Evaluation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{experiment_name} - Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Plot accuracy
    plt.figure(figsize=(9, 5))
    plt.plot(
        np.array(history["train_accuracy"]) * 100,
        label="Training Accuracy"
    )
    plt.plot(
        np.array(history["eval_accuracy"]) * 100,
        label="Evaluation Accuracy"
    )
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title(f"{experiment_name} - Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Release GPU memory
    del model
    del optimizer
    del scaler

    torch.cuda.empty_cache()

    return results, history

In [ ]:
baseline_results, baseline_history = run_experiment(
    experiment_name="Baseline ResNet10",
    use_batchnorm=False,
    dropout_p=0.0,
    weight_decay=0.0
)


Starting: Baseline ResNet10
Epoch [1/300] | Time: 29.37s | Train Loss: 2.5331 | Train Acc: 28.33% | Eval Loss: 1.4689 | Eval Acc: 43.29%
Epoch [2/300] | Time: 24.39s | Train Loss: 1.2749 | Train Acc: 52.89% | Eval Loss: 1.0957 | Eval Acc: 61.35%
Epoch [3/300] | Time: 23.66s | Train Loss: 0.9551 | Train Acc: 65.76% | Eval Loss: 0.9210 | Eval Acc: 67.76%
Epoch [4/300] | Time: 23.53s | Train Loss: 0.7587 | Train Acc: 73.18% | Eval Loss: 0.7999 | Eval Acc: 71.86%
Epoch [5/300] | Time: 23.57s | Train Loss: 0.6217 | Train Acc: 78.18% | Eval Loss: 0.7199 | Eval Acc: 75.88%
Epoch [6/300] | Time: 23.97s | Train Loss: 0.5109 | Train Acc: 81.91% | Eval Loss: 0.6791 | Eval Acc: 77.11%
Epoch [7/300] | Time: 23.75s | Train Loss: 0.4175 | Train Acc: 85.56% | Eval Loss: 0.7006 | Eval Acc: 77.24%
Epoch [8/300] | Time: 23.64s | Train Loss: 0.3350 | Train Acc: 88.16% | Eval Loss: 0.7155 | Eval Acc: 77.87%
Epoch [9/300] | Time: 23.62s | Train Loss: 0.2714 | Train Acc: 90.57% | Eval Loss: 0.8506 | Eval Ac